# M1: Ingesta TCGA-BRCA - GEN-MAMA-MX

Este notebook automatiza la descarga y procesamiento de datos de mutaciones somáticas del proyecto **TCGA-BRCA** (Breast Invasive Carcinoma) utilizando la API del **GDC (Genomic Data Commons)**.

**Objetivos:**
1. Descargar datos de mutaciones somáticas filtrados para variantes missense.
2. Identificar los genes más frecuentemente mutados.
3. Priorizar las variantes para el análisis estructural (Módulo M3).

## 0. Configuración del Entorno (Colab)

Si estás ejecutando este notebook en Google Colab, la siguiente celda configurará automáticamente las carpetas necesarias y recreará los scripts de soporte.

In [ ]:
import os
import subprocess

IS_COLAB = 'google.colab' in str(get_ipython())

if IS_COLAB:
    print("Entorno Google Colab detectado. Configurando carpetas...")
    os.makedirs("scripts", exist_ok=True)
    os.makedirs("data/fasta", exist_ok=True)
    os.makedirs("data/structures", exist_ok=True)
    
    # Escribir el script de descarga de FASTA
    fasta_script = 'import requests\nimport os\nimport json\nimport io\nimport re\nfrom Bio import SeqIO\nfrom Bio.Seq import MutableSeq\n\nFASTA_DIR = "data/fasta"\nos.makedirs(FASTA_DIR, exist_ok=True)\n\ndef get_uniprot_id(gene_symbol):\n    """Search UniProt for the best accession ID for a human gene symbol."""\n    url = "https://rest.uniprot.org/uniprotkb/search"\n    params = {\n        "query": f\'gene:{gene_symbol} AND organism_id:9606 AND reviewed:true\',\n        "fields": "accession",\n        "format": "json",\n        "size": 1\n    }\n    try:\n        res = requests.get(url, params=params)\n        if res.status_code == 200:\n            data = res.json()\n            if data.get("results"):\n                return data["results"][0]["primaryAccession"]\n    except Exception as e:\n        print(f"Error fetching UniProt ID for {gene_symbol}: {e}")\n    return None\n\ndef fetch_fasta(accession):\n    """Fetch the FASTA sequence from UniProt."""\n    url = f"https://www.uniprot.org/uniprot/{accession}.fasta"\n    try:\n        res = requests.get(url)\n        if res.status_code == 200:\n            return res.text\n    except Exception as e:\n        print(f"Error fetching FASTA for {accession}: {e}")\n    return None\n\ndef apply_mutation(wt_fasta, protein_change):\n    """Apply a missense mutation to a FASTA sequence."""\n    try:\n        record = SeqIO.read(io.StringIO(wt_fasta), "fasta")\n        wt_seq = record.seq\n        match = re.search(r"([A-Z])(\\d+)([A-Z])", protein_change)\n        if match:\n            wt_aa, pos, mut_aa = match.group(1), int(match.group(2)), match.group(3)\n            idx = pos - 1\n            if idx < len(wt_seq) and wt_seq[idx] == wt_aa:\n                mut_seq = MutableSeq(wt_seq)\n                mut_seq[idx] = mut_aa\n                return str(wt_seq), str(mut_seq)\n    except Exception as e:\n        print(f"Error applying mutation {protein_change}: {e}")\n    return None, None\n\ndef main():\n    if not os.path.exists("top_variants.json"):\n        print("Error: top_variants.json not found.")\n        return\n\n    with open("top_variants.json", \'r\') as f:\n        variants = json.load(f)\n\n    print(f"Processing {len(variants)} variants...")\n    for var in variants:\n        gene = var.get(\'Gene\')\n        change = var.get(\'AA_Change\')\n        if not gene or not change or change == "N/A":\n            continue\n        \n        print(f"Fetching sequences for {gene} {change}...")\n        acc = get_uniprot_id(gene)\n        if acc:\n            wt_fasta = fetch_fasta(acc)\n            if wt_fasta:\n                wt_str, mut_str = apply_mutation(wt_fasta, change)\n                if wt_str and mut_str:\n                    with open(f"{FASTA_DIR}/{gene}_WT.fasta", "w") as f:\n                        f.write(f">{gene}_WT\\n{wt_str}\\n")\n                    with open(f"{FASTA_DIR}/{gene}_{change}.fasta", "w") as f:\n                        f.write(f">{gene}_{change}\\n{mut_str}\\n")\n                    print(f"Saved FASTA for {gene} {change}")\n\nif __name__ == "__main__":\n    main()\n'
    with open("scripts/fetch_fasta_colab.py", "w") as f:
        f.write(fasta_script)
    
    # Escribir el script de predicción automáticamente
    predict_script = 'import os\nimport json\nimport subprocess\n\n# --- CONFIGURATION ---\nIS_COLAB = os.path.exists("/content")\nBASE_PATH = "/content" if IS_COLAB else "."\n\nDATA_DIR = os.path.join(BASE_PATH, "data")\nFASTA_DIR = os.path.join(DATA_DIR, "fasta")\nSTRUCTURES_DIR = os.path.join(DATA_DIR, "structures")\nTOP_VARIANTS_FILE = os.path.join(BASE_PATH, "top_variants.json")\n\ndef setup_boltz():\n    print("Installing Boltz-1...")\n    subprocess.run(["pip", "install", "boltz"], check=True)\n    print("Boltz-1 installed.")\n\ndef predict_variant(hugo_symbol, protein_change):\n    wt_fasta = os.path.join(FASTA_DIR, f"{hugo_symbol}_WT.fasta")\n    mut_fasta = os.path.join(FASTA_DIR, f"{hugo_symbol}_{protein_change}.fasta")\n    \n    for fasta_path, tag in [(wt_fasta, "WT"), (mut_fasta, "MUT")]:\n        if not os.path.exists(fasta_path):\n            print(f"Skipping {tag}: {fasta_path} not found.")\n            continue\n            \n        output_name = f"{hugo_symbol}_{protein_change}_{tag}"\n        print(f"Predicting structure for {output_name}...")\n        \n        try:\n            subprocess.run([\n                "boltz", "predict", \n                fasta_path,\n                "--out_dir", STRUCTURES_DIR,\n                "--name", output_name\n            ], check=True)\n            print(f"Successfully predicted {output_name}")\n        except Exception as e:\n            print(f"Error predicting {output_name}: {e}")\n\ndef main():\n    if not os.path.exists(STRUCTURES_DIR):\n        os.makedirs(STRUCTURES_DIR)\n        \n    if not os.path.exists(TOP_VARIANTS_FILE):\n        print(f"Error: {TOP_VARIANTS_FILE} not found.")\n        return\n\n    with open(TOP_VARIANTS_FILE, \'r\') as f:\n        variants = json.load(f)\n    \n    try:\n        subprocess.run(["boltz", "--version"], check=True, capture_output=True)\n    except:\n        setup_boltz()\n\n    for v in variants:\n        gene = v.get(\'Gene\')\n        change = v.get(\'AA_Change\')\n        if gene and change and change != "N/A":\n            predict_variant(gene, change)\n\nif __name__ == "__main__":\n    main()\n'
    with open("scripts/predict_structures.py", "w") as f:
        f.write(predict_script)
    
    # Instalar dependencias necesarias
    print("Instalando dependencias (biopython)...")
    subprocess.run(["pip", "install", "biopython"], check=True)
    
    print("Configuración completada. Scripts creados en 'scripts/'.")
else:
    print("Entorno local detectado. Asegúrate de tener la carpeta 'scripts' y 'data' configuradas.")

## 0. Configuración del Entorno (Colab)

Si estás ejecutando este notebook en Google Colab, la siguiente celda configurará automáticamente las carpetas necesarias y recreará los scripts de soporte.

In [ ]:
import os
import subprocess

IS_COLAB = 'google.colab' in str(get_ipython())

if IS_COLAB:
    print("Entorno Google Colab detectado. Configurando carpetas...")
    os.makedirs("scripts", exist_ok=True)
    os.makedirs("data/fasta", exist_ok=True)
    os.makedirs("data/structures", exist_ok=True)
    
    # Escribir el script de descarga de FASTA
    fasta_script = 'import requests\nimport os\nimport json\nimport io\nimport re\nfrom Bio import SeqIO\nfrom Bio.Seq import MutableSeq\n\nFASTA_DIR = "data/fasta"\nos.makedirs(FASTA_DIR, exist_ok=True)\n\ndef get_uniprot_id(gene_symbol):\n    """Search UniProt for the best accession ID for a human gene symbol."""\n    url = "https://rest.uniprot.org/uniprotkb/search"\n    params = {\n        "query": f\'gene:{gene_symbol} AND organism_id:9606 AND reviewed:true\',\n        "fields": "accession",\n        "format": "json",\n        "size": 1\n    }\n    try:\n        res = requests.get(url, params=params)\n        if res.status_code == 200:\n            data = res.json()\n            if data.get("results"):\n                return data["results"][0]["primaryAccession"]\n    except Exception as e:\n        print(f"Error fetching UniProt ID for {gene_symbol}: {e}")\n    return None\n\ndef fetch_fasta(accession):\n    """Fetch the FASTA sequence from UniProt."""\n    url = f"https://www.uniprot.org/uniprot/{accession}.fasta"\n    try:\n        res = requests.get(url)\n        if res.status_code == 200:\n            return res.text\n    except Exception as e:\n        print(f"Error fetching FASTA for {accession}: {e}")\n    return None\n\ndef apply_mutation(wt_fasta, protein_change):\n    """Apply a missense mutation to a FASTA sequence."""\n    try:\n        record = SeqIO.read(io.StringIO(wt_fasta), "fasta")\n        wt_seq = record.seq\n        match = re.search(r"([A-Z])(\\d+)([A-Z])", protein_change)\n        if match:\n            wt_aa, pos, mut_aa = match.group(1), int(match.group(2)), match.group(3)\n            idx = pos - 1\n            if idx < len(wt_seq) and wt_seq[idx] == wt_aa:\n                mut_seq = MutableSeq(wt_seq)\n                mut_seq[idx] = mut_aa\n                return str(wt_seq), str(mut_seq)\n    except Exception as e:\n        print(f"Error applying mutation {protein_change}: {e}")\n    return None, None\n\ndef main():\n    if not os.path.exists("top_variants.json"):\n        print("Error: top_variants.json not found.")\n        return\n\n    with open("top_variants.json", \'r\') as f:\n        variants = json.load(f)\n\n    print(f"Processing {len(variants)} variants...")\n    for var in variants:\n        gene = var.get(\'Gene\')\n        change = var.get(\'AA_Change\')\n        if not gene or not change or change == "N/A":\n            continue\n        \n        print(f"Fetching sequences for {gene} {change}...")\n        acc = get_uniprot_id(gene)\n        if acc:\n            wt_fasta = fetch_fasta(acc)\n            if wt_fasta:\n                wt_str, mut_str = apply_mutation(wt_fasta, change)\n                if wt_str and mut_str:\n                    with open(f"{FASTA_DIR}/{gene}_WT.fasta", "w") as f:\n                        f.write(f">{gene}_WT\\n{wt_str}\\n")\n                    with open(f"{FASTA_DIR}/{gene}_{change}.fasta", "w") as f:\n                        f.write(f">{gene}_{change}\\n{mut_str}\\n")\n                    print(f"Saved FASTA for {gene} {change}")\n\nif __name__ == "__main__":\n    main()\n'
    with open("scripts/fetch_fasta_colab.py", "w") as f:
        f.write(fasta_script)
    
    # Escribir el script de predicción automáticamente
    predict_script = 'import os\nimport json\nimport subprocess\n\n# --- CONFIGURATION ---\nIS_COLAB = os.path.exists("/content")\nBASE_PATH = "/content" if IS_COLAB else "."\n\nDATA_DIR = os.path.join(BASE_PATH, "data")\nFASTA_DIR = os.path.join(DATA_DIR, "fasta")\nSTRUCTURES_DIR = os.path.join(DATA_DIR, "structures")\nTOP_VARIANTS_FILE = os.path.join(BASE_PATH, "top_variants.json")\n\ndef setup_boltz():\n    print("Installing Boltz-1...")\n    subprocess.run(["pip", "install", "boltz"], check=True)\n    print("Boltz-1 installed.")\n\ndef predict_variant(hugo_symbol, protein_change):\n    wt_fasta = os.path.join(FASTA_DIR, f"{hugo_symbol}_WT.fasta")\n    mut_fasta = os.path.join(FASTA_DIR, f"{hugo_symbol}_{protein_change}.fasta")\n    \n    for fasta_path, tag in [(wt_fasta, "WT"), (mut_fasta, "MUT")]:\n        if not os.path.exists(fasta_path):\n            print(f"Skipping {tag}: {fasta_path} not found.")\n            continue\n            \n        output_name = f"{hugo_symbol}_{protein_change}_{tag}"\n        print(f"Predicting structure for {output_name}...")\n        \n        try:\n            subprocess.run([\n                "boltz", "predict", \n                fasta_path,\n                "--out_dir", STRUCTURES_DIR,\n                "--name", output_name\n            ], check=True)\n            print(f"Successfully predicted {output_name}")\n        except Exception as e:\n            print(f"Error predicting {output_name}: {e}")\n\ndef main():\n    if not os.path.exists(STRUCTURES_DIR):\n        os.makedirs(STRUCTURES_DIR)\n        \n    if not os.path.exists(TOP_VARIANTS_FILE):\n        print(f"Error: {TOP_VARIANTS_FILE} not found.")\n        return\n\n    with open(TOP_VARIANTS_FILE, \'r\') as f:\n        variants = json.load(f)\n    \n    try:\n        subprocess.run(["boltz", "--version"], check=True, capture_output=True)\n    except:\n        setup_boltz()\n\n    for v in variants:\n        gene = v.get(\'Gene\')\n        change = v.get(\'AA_Change\')\n        if gene and change and change != "N/A":\n            predict_variant(gene, change)\n\nif __name__ == "__main__":\n    main()\n'
    with open("scripts/predict_structures.py", "w") as f:
        f.write(predict_script)
    
    # Instalar dependencias necesarias
    print("Instalando dependencias (biopython)...")
    subprocess.run(["pip", "install", "biopython"], check=True)
    
    print("Configuración completada. Scripts creados en 'scripts/'.")
else:
    print("Entorno local detectado. Asegúrate de tener la carpeta 'scripts' y 'data' configuradas.")

In [2]:
import pandas as pd
import requests
import json
import os
from tqdm import tqdm

print("Librerías cargadas correctamente.")

Librerías cargadas correctamente.


## 1. Configuración de la API del GDC

Definimos los parámetros para consultar el portal GDC y obtener las mutaciones somáticas (SSM) del proyecto BRCA.

In [3]:
def fetch_ssms(project_id="TCGA-BRCA", size=1000):
    """Consulta la API de GDC para obtener Simple Somatic Mutations."""
    ssm_url = "https://api.gdc.cancer.gov/ssms"
    
    # Filtros: Proyecto TCGA-BRCA y Variantes Missense
    filters = {
        "op": "and",
        "content": [
            {
                "op": "in",
                "content": {
                    "field": "occurrence.case.project.project_id",
                    "value": [project_id]
                }
            },
            {
                "op": "in",
                "content": {
                    "field": "consequence.transcript.consequence_type",
                    "value": ["missense_variant"]
                }
            }
        ]
    }
    
    params = {
        "filters": json.dumps(filters),
        "fields": "genomic_dna_change,consequence.transcript.gene.symbol,consequence.transcript.aa_change,occurrence.case.project.project_id",
        "format": "JSON",
        "size": size
    }
    
    response = requests.get(ssm_url, params=params)
    return response.json()

print("Iniciando descarga de datos de GDC...")
data = fetch_ssms(size=2000)
print(f"Descarga completada. Registros obtenidos: {len(data['data']['hits'])}")

Iniciando descarga de datos de GDC...
Descarga completada. Registros obtenidos: 2000


## 2. Procesamiento y Limpieza de Datos

Transformamos el JSON en un DataFrame de Pandas para su manipulación.

In [4]:
hits = data['data']['hits']
rows = []

for hit in hits:
    gene_symbol = hit['consequence'][0]['transcript']['gene']['symbol']
    aa_change = hit['consequence'][0]['transcript'].get('aa_change', 'N/A')
    dna_change = hit['genomic_dna_change']
    
    rows.append({
        "Gene": gene_symbol,
        "AA_Change": aa_change,
        "DNA_Change": dna_change
    })

df = pd.DataFrame(rows)
df.dropna(inplace=True)

# Guardar dataset completo
df.to_csv("mutations_brca.csv", index=False)
df.head(10)

,Gene,AA_Change,DNA_Change
0,SEPTIN14,I12M,chr7:g.55861961T>C
1,CLK1,N/A,chr2:g.200861318A>C
2,KCNJ16,E315Q,chr17:g.70133030G>C
3,CX3CR1,N/A,chr3:g.39281685G>A
4,CNKSR2,N/A,chrX:g.21432763C>T
5,ADH7,N/A,chr4:g.99420574C>T
6,APC,A1366V,chr5:g.112839691C>T
7,SEMA4B,N203H,chr15:g.90228002A>C
8,SYT16,N/A,chr14:g.62081147G>A
9,CSMD1,F458V,chr8:g.3493702A>C


## 3. Priorización de Variantes (M2)

Calculamos la frecuencia de las mutaciones para identificar los hotspots más relevantes del proyecto.

In [5]:
top_variants = df.groupby(['Gene', 'AA_Change']).size().reset_index(name='Frequency')
top_variants = top_variants.sort_values(by='Frequency', ascending=False).head(10)

print("Top 10 Variantes Missense en TCGA-BRCA:")
print(top_variants)

# Guardar resultados para el siguiente módulo
top_variants_dict = top_variants.to_dict(orient='records')
with open("top_variants.json", "w") as f:
    json.dump(top_variants_dict, f, indent=4)

print("\nArchivos 'mutations_brca.csv' y 'top_variants.json' generados.")

Top 10 Variantes Missense en TCGA-BRCA:
         Gene AA_Change  Frequency
132      ARSK       N/A          3
648    GNPTAB       N/A          2
18     ACADSB       N/A          2
607     FOXA1       N/A          2
1499  SHROOM3       N/A          2
958      MEI1       N/A          2
1622    SUGP2       N/A          2
1208   PEX11B       N/A          2
536     ERCC6       N/A          2
602      FMN2       N/A          2

Archivos 'mutations_brca.csv' y 'top_variants.json' generados.


## 3.5 Generación de Secuencias FASTA

Este paso descarga las secuencias de UniProt para los genes priorizados y aplica las mutaciones correspondientes para generar los archivos `.fasta` necesarios para Boltz-1.

In [ ]:
# Ejecutar descarga de FASTA
!python scripts/fetch_fasta_colab.py

## 4. Predicción de Estructuras (Boltz-1)

Este bloque ejecuta la predicción 3D de forma automática. 

**Nota:** Si ejecutaste la celda de **Configuración del Entorno** al inicio, el script ya debería estar en su lugar.

In [ ]:
import os
# Intentar localizar el script en el entorno de Colab o Local
script_path = "scripts/predict_structures.py"

if os.path.exists(script_path):
    print(f"Ejecutando script desde: {script_path}")
    !python {script_path}
else:
    print("ERROR: No se encontró 'scripts/predict_structures.py'.")
    print("Por favor, revisa que la carpeta 'scripts' exista o ejecuta la sección de Configuración al inicio del notebook.")